# Gun Detection Training (YOLOv8 + MLflow)

This notebook handles the end-to-end process for Gun Detection:
1. **Data Preparation**: Merges Real and Synthetic datasets (V2/V3), splits them, and formats for YOLO.
2. **Training**: Fine-tunes a YOLOv8 model with MLflow tracking.
3. **Validation**: Evaluates the model on the test split.

In [1]:
# --- 1. DATA PREPARATION ---
import sys
import os

# Ensure src/ is in python path
notebook_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(notebook_dir, '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

from src.prepare_data import main as prepare_data

print("Running Data Preparation...")
# This will:
# 1. Gather files from Real, SynV2, and SynV3 datasets
# 2. Split them into Train/Val/Test (70/20/10)
# 3. Create 'data/images' and 'data/labels' structure
# 4. Generate 'data.yaml'
prepare_data()

Running Data Preparation...
Project Root: D:\_pribadi\gun_detection
Found 2384 pairs in Real Dataset
Found 23 pairs in Syn V2 Dataset
Found 1842 pairs in Syn V3 Dataset

DATASET              | TRAIN      | VAL        | TEST       | TOTAL     
--------------------------------------------------------------------------------
Real Data            | 1668       | 476        | 240        | 2384      
Synthetic V2         | 16         | 4          | 3          | 23        
Synthetic V3         | 1289       | 368        | 185        | 1842      
--------------------------------------------------------------------------------
COMBINED TOTAL       | 2973       | 848        | 428        | 4249      



Processing test: 100%|██████████| 428/428 [00:02<00:00, 154.38it/s]


Created data_real.yaml
Created data_syn_v2.yaml
Created data_syn_v3.yaml
Created data_real_syn_v2.yaml
Created data_real_syn_v3.yaml
Created data_combined.yaml
Created data.yaml
Data preparation complete.


In [1]:
# --- 2. MODEL TRAINING ---
import os
import sys
import mlflow
import mlflow.pytorch
import mlflow.data
import torch
import pandas as pd
from ultralytics import YOLO, settings

notebook_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(notebook_dir, '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

# Configuration
mlruns_dir = os.path.join(project_root, "mlruns")
# tracking_uri = f"file:///{mlruns_dir}" # OLD
# NEW: SQLite DB
db_path = os.path.join(project_root, "mlflow.db")
tracking_uri = f"sqlite:///{db_path}"

model_name = "yolo26s"  # Or 'yolov8n', 'yolov8s', etc.

# MLflow Setup
mlflow.set_tracking_uri(tracking_uri)
os.environ["MLFLOW_TRACKING_URI"] = tracking_uri
os.environ["MLFLOW_EXPERIMENT_NAME"] = "Gun_Detection_Experiment"
settings.update({"mlflow": True})

print(f"Experiment: {os.environ['MLFLOW_EXPERIMENT_NAME']}")
print(f"Tracking URI: {tracking_uri}")

# Initialize Model
model = YOLO(model_name)

dataset_names = ["real", "syn_v2", "syn_v3", "real_syn_v2", "real_syn_v3", "combined"]
selected_dataset = dataset_names[0]

# Data Config Path (Generated by prepare_data)
data_config = os.path.join(project_root, "data", f"data_{selected_dataset}.yaml")


print(f"Training with config: {data_config}")

# Train
results = model.train(
    data=data_config,
    epochs=200,
    imgsz=640,
    batch=16,
    device=0 if torch.cuda.is_available() else 'cpu',
    workers=4,
    project=os.path.join(project_root, "runs/detect"),
    name=f"gun_detection_run_{selected_dataset}",
    exist_ok=True,
    verbose=True
)

# --- Explicit MLflow Logging ---
# Ultralytics likely closed the run. We try to resume it or use the active one.
print("Appending explicit logs to MLflow run...")

try:
    # Check if a run is active, otherwise try to get the last one
    run_to_use = mlflow.active_run()
    if run_to_use is None:
        run_to_use = mlflow.last_active_run()
    
    if run_to_use:
        run_id = run_to_use.info.run_id
        print(f"Using Run ID: {run_id}")
        
        with mlflow.start_run(run_id=run_id):
            # 1. Log Dataset
            print("Logging dataset...")
            try:
                dataset_source = pd.DataFrame([{
                    "name": selected_dataset,
                    "config_path": data_config,
                    "project": "gun_detection"
                }])
                dataset = mlflow.data.from_pandas(dataset_source, name=selected_dataset)
                mlflow.log_input(dataset, context="training")
            except Exception as e:
                print(f"Failed to log dataset: {e}")

            # 2. Log Model
            print("Logging model artifact...")
            try:
                best_model_path = os.path.join(results.save_dir, "weights", "best.pt")
                if os.path.exists(best_model_path):
                    print(f"Found best model at: {best_model_path}")
                    # Load as YOLO model to get the underlying PyTorch model
                    # Explicitly load weights to ensure we have the trained ones
                    best_model = YOLO(best_model_path)
                    # Log the PyTorch model artifact
                    mlflow.pytorch.log_model(
                        best_model.model,
                        "model",
                        pip_requirements=["ultralytics", "torch", "torchvision", "pandas", "mlflow"]
                    )
                    print("Model logged successfully.")
                else:
                    print(f"Best model not found at {best_model_path}")
            except Exception as e:
                print(f"Failed to log model: {e}")
    else:
        print("No active or recent MLflow run found to attach logs to.")

except Exception as e:
    print(f"An error occurred during manual logging: {e}")


Experiment: Gun_Detection_Experiment
Tracking URI: file:///d:\_pribadi\gun_detection\mlruns
Training with config: d:\_pribadi\gun_detection\data\data_real.yaml
New https://pypi.org/project/ultralytics/8.4.14 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.12  Python-3.10.11 torch-2.10.0+cu126 CUDA:0 (NVIDIA GeForce RTX 4070 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=d:\_pribadi\gun_detection\data\data_real.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=

d:\_pribadi\gun_detection\.venv\lib\site-packages\mlflow\tracking\_tracking_service\utils.py:178: FutureWarning: The filesystem tracking backend (e.g., './mlruns') will be deprecated in February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://github.com/mlflow/mlflow/issues/18534 for more details and migration guidance. For migrating existing data, https://github.com/mlflow/mlflow-export-import can be used.
  return FileStore(store_uri, store_uri)


MLflow: logging run_id(cc1156d754ce460b91ca4336548b6fed) to file:///d:\_pribadi\gun_detection\mlruns
MLflow: disable with 'yolo settings mlflow=False'
Image sizes 640 train, 640 val
Using 4 dataloader workers
Logging results to D:\_pribadi\gun_detection\runs\detect\gun_detection_run_real
Starting training for 200 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      1/200      4.72G      1.438      2.345    0.02273          9        640: 100% ━━━━━━━━━━━━ 105/105 2.4it/s 43.2s0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 15/15 1.3it/s 11.9s.3s
                   all        476        479      0.795      0.724      0.769      0.359

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      2/200      4.75G      1.761      1.481    0.03041          8        640: 100% ━━━━━━━━━━━━ 105/105 4.2it/s 25.3s0.2s
                 Class     Images  Instances      B

In [ ]:
# --- 3. VALIDATION ---
print("Validating on Test set...")
metrics = model.val(split='test')
print(f"mAP50-95: {metrics.box.map}")